# Multi-Source Data Engineering Pipeline for District-Level Boro Rice in Bangladesh

**Paper:** *An open, tested data engineering framework for integrating official crop statistics with satellite and
climate data: district-level Boro rice in Bangladesh* (Environmental Modelling & Software, submitted)
**Authors:** Tasnim Ahmad Mumu (Department of Agronomy and Agricultural Extension, University of Rajshahi) and
Rezaul Karim (Department of Statistics and Data Science, University of Rajshahi; corresponding author)
**Period:** harvest years 2011–2025 (BBS crop years 2010-11 to 2024-25) for every dataset
**Districts:** all 64 districts of Bangladesh

---

## Abstract

This notebook builds the complete dataset of the paper from the original sources, step by step. It extracts the
district Boro rice statistics from 14 editions (2012–2025) of the Bangladesh Bureau of Statistics (BBS) *Yearbook of
Agricultural Statistics*, checks every value against the redundancy within the yearbooks, extracts cropland NDVI,
ERA5-Land climate and two flood records (JRC monthly water history and Sentinel-1 radar) for each district from
Google Earth Engine, and integrates everything by harvest year in a DuckDB database. No value is estimated,
interpolated or imputed: a district-year enters the dataset only if BBS printed it, and every value carries the
yearbook edition and page from which it was read.

The data are organised in three layers:

| Layer | Content | Folder |
|---|---|---|
| **Bronze** | sources as received: parsed yearbook rows, Earth Engine district extracts, a manifest of every file | `build/data/bronze/` |
| **Silver** | one validated table per source, keyed on district and harvest year (Parquet) | `build/data/silver/` |
| **Gold** | integrated analytical tables in DuckDB, including the district-year panel | `build/data/gold/` |

The last two sections run the full validation report on the data built here and confirm that they are identical to
the data produced by the command-line pipeline (`v2/run_all.py`).

---

## Contents

0. Environment setup and configuration
1. Bronze layer: raw data ingestion
   * 1a. GADM v4.1 district boundaries
   * 1b. BBS yearbooks: table extraction from the PDFs
   * 1c. BBS: row classification and unit identities
   * 1d. BBS: national totals, edition comparison, region rows and the district-year table
   * 1e. Google Earth Engine: setup
   * 1f. Earth Engine extraction functions (NDVI, ERA5-Land, JRC, Sentinel-1)
   * 1g. Running the extraction with a cache and a completeness test
   * 1h. Metadata manifest
2. Silver layer: input contract, district harmonisation and validation
3. Gold layer: DuckDB analytical tables
4. Validation report
5. Comparison with the command-line pipeline

### Inputs that must be downloaded first

* **BBS yearbooks** (not redistributed): the 2012–2025 editions of the *Yearbook of Agricultural Statistics* from
  bbs.gov.bd, saved as `data/bronze/bbs/pdfs/bbs_agri_yearbook_<edition>.pdf` in the repository root.
* **GADM v4.1** boundaries: downloaded automatically by cell 1a if missing (the licence does not permit
  redistribution).
* **Google Earth Engine**: an account with access to a Cloud project, authenticated once with
  `earthengine authenticate`. Needed only to compute extracts that are not in the cache (see 1g).

---
## 0. Environment Setup

All imports for the pipeline are here. The packages and their exact versions are listed in
`v2/requirements-lock.txt`; install them once with `python -m pip install -r v2/requirements-lock.txt`.

In [1]:
# ── Install packages (run once, then comment out) ─────────────────────────
# !pip install -r ../requirements-lock.txt

# ── Standard library ──────────────────────────────────────────────────────
import hashlib
import json
import os
import re
import shutil
import subprocess
import sys
import time
import urllib.request
import warnings
import zipfile
from datetime import datetime, timezone
from importlib.metadata import version
from pathlib import Path

# ── Third party ───────────────────────────────────────────────────────────
import camelot                    # PDF table extraction (lattice mode)
import duckdb                     # embedded analytical database (Gold layer)
import geopandas as gpd           # district boundaries
import numpy as np
import pandas as pd
from IPython.display import display

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 200)
print("Python", sys.version.split()[0], "| pandas", pd.__version__, "| duckdb", duckdb.__version__,
      "| camelot", version("camelot-py"), "| earthengine-api", version("earthengine-api"))

Python 3.13.7 | pandas 2.3.3 | duckdb 1.5.2 | camelot 1.0.9 | earthengine-api 1.7.22


### 0b. Paths and run options

* `ROOT` is the repository root; the downloaded inputs (yearbook PDFs, GADM) live in `ROOT/data/bronze/`.
* Everything this notebook produces is written to `v2/notebooks/build/`, laid out like the `v2` folder of the
  command-line pipeline, so the two can be compared file by file (Section 5).
* `REUSE_REPOSITORY_EXTRACTS = True` copies the Earth Engine district extracts already in the repository into the
  build cache, so the notebook runs in a few minutes. Set it to `False` to compute all 53 extracts from Earth Engine
  (about 75 minutes). Either way, cell 1g recomputes one extract live and compares it with the cache.

In [2]:
NB_DIR = Path.cwd()
V2 = next(p for p in [NB_DIR, *NB_DIR.parents] if (p / "run_all.py").exists())   # the v2 folder
ROOT = V2.parent                                                                  # repository root

INPUT = ROOT / "data" / "bronze"                 # downloaded inputs: yearbook PDFs, GADM shapefiles
PDF_DIR = INPUT / "bbs" / "pdfs"
SHP_DIR = INPUT / "gadm_shapefiles"

BUILD = V2 / "notebooks" / "build"               # everything this notebook writes
BRONZE = BUILD / "data" / "bronze"
BBS_OUT, GEE_OUT = BRONZE / "bbs", BRONZE / "gee"
SILVER = BUILD / "data" / "silver"
GOLD_DB = BUILD / "data" / "gold" / "bangladesh_agri_v2.duckdb"
for folder in [BBS_OUT, GEE_OUT, SILVER, GOLD_DB.parent]:
    folder.mkdir(parents=True, exist_ok=True)

GEE_PROJECT = "bd-agri-climate"                  # ← your Google Earth Engine Cloud project
REUSE_REPOSITORY_EXTRACTS = True                 # False = compute every extract from Earth Engine (~75 min)
CHECK_ONE_EXTRACT_LIVE = True                    # recompute one extract and compare it with the cache

print("inputs:", INPUT.relative_to(ROOT), "| build:", BUILD.relative_to(ROOT))

inputs: data\bronze | build: v2\notebooks\build


### 0c. Study period and the harvest-year convention

A Boro season is identified by its **harvest year** $t$: the crop grown from November of year $t-1$ to May of year
$t$, which BBS reports as crop year "$(t-1)$–$t$". Crop year 2010-11 is harvest year 2011, and crop year 2024-25 is
harvest year 2025. Every table below is keyed on harvest year, so that statistics, satellite and climate data refer
to the same season.

Each satellite and climate variable is computed over a date **window** relative to $t$, written
`(offset, first month, last month)`. If the first month is later than the last, the window runs into the next year;
for example `(-1, 11, 5)` is November $t-1$ to May $t$.

In [3]:
HARVEST_YEARS = list(range(2011, 2026))          # 2011–2025: 15 Boro seasons


def window(t, offset, m0, m1):
    # Start (inclusive) and end (exclusive) dates of a window for harvest year t
    y0 = t + offset
    y1 = y0 if m1 >= m0 else y0 + 1
    ey, em = (y1 + 1, 1) if m1 == 12 else (y1, m1 + 1)
    return f"{y0}-{m0:02d}-01", f"{ey}-{em:02d}-01"


print("Boro season of harvest year 2017:", window(2017, -1, 11, 5))
print("Flash-flood window of 2017:      ", window(2017, 0, 4, 5))

Boro season of harvest year 2017: ('2016-11-01', '2017-06-01')
Flash-flood window of 2017:       ('2017-04-01', '2017-06-01')


### 0d. Extraction settings

Every setting of the satellite and climate extraction is declared here once and used by the extraction functions
and the manifest:

* **NDVI**: MODIS MOD13Q1 v061, mean (`ndvi_peak`) and maximum (`ndvi_max`) of the 16-day composites from January to
  April of year $t$; pixels with SummaryQA ≤ 1 only, and only 250 m cells in which ESA WorldCover cropland covers
  more than half the cell.
* **Climate**: ERA5-Land daily aggregates; season (November $t-1$ to May $t$) mean temperature, mean daily maximum and
  total precipitation; hot days with a daily maximum above 35 °C from March to May.
* **JRC flood**: monthly water history (Landsat, 30 m); water in any month of the window and not in the matching dry
  window (January–March); the share of each district with at least one valid observation is kept with it.
* **Sentinel-1 flood**: VV backscatter (IW mode), monthly median, 50 m focal median, water below −18 dB, slopes above
  5° excluded.

**Coverage.** A variable is extracted for every harvest year in which its product provides the required months,
and is left missing otherwise. The JRC monthly water history ends in December 2021; Sentinel-1 data over Bangladesh
begin in October 2014, so the first complete monsoon season is 2015 (harvest year 2016).

In [4]:
EXTRACTS = {
    "ndvi": dict(collection="MODIS/061/MOD13Q1", version="Collection 6.1 (061)",
                 bands="NDVI, SummaryQA", scale_m=250, windows={"season": (0, 1, 4)},
                 mask="ESA/WorldCover/v200 (2021) class 40 covering > 50 % of the MODIS cell; SummaryQA ≤ 1",
                 method="mean (ndvi_peak) and maximum (ndvi_max) of the 16-day composites in the window; "
                        "district mean over masked pixels"),
    "climate": dict(collection="ECMWF/ERA5_LAND/DAILY_AGGR", version="ERA5-Land daily aggregates",
                    bands="temperature_2m, temperature_2m_max, total_precipitation_sum", scale_m=1000,
                    windows={"season": (-1, 11, 5), "hot": (0, 3, 5)}, hot_threshold_c=35,
                    mask="none (land-only product)",
                    method="season mean of daily mean and daily maximum temperature; season total precipitation; "
                           "number of days with daily maximum > 35 °C in the hot window; district mean"),
    "jrc": dict(collection="JRC/GSW1_4/MonthlyHistory", version="v1.4", bands="water", scale_m=30,
                windows={"monsoon": (-1, 6, 10), "monsoon_dry": (-1, 1, 3),
                         "flash": (0, 4, 5), "flash_dry": (0, 1, 3)},
                mask="none",
                method="water (class 2) in any month of the window and not water in any month of the "
                       "matching dry window; share of district area; *_obs = share with ≥ 1 valid observation"),
    "s1": dict(collection="COPERNICUS/S1_GRD", version="GRD, IW mode, VV polarisation", bands="VV", scale_m=60,
               windows={"monsoon": (-1, 6, 10), "monsoon_dry": (-1, 1, 3),
                        "flash": (0, 4, 5), "flash_dry": (0, 1, 3)},
               threshold_db=-18, focal_median_m=50, slope_max_deg=5,
               mask="slope < 5° from USGS/SRTMGL1_003",
               method="monthly median VV, 50 m focal median, water if < −18 dB; water in any month of the "
                      "window and not water in the dry-window median; share of district area"),
}

# variable -> (extract, first harvest year, last harvest year)
COVERAGE = {
    "ndvi_peak": ("ndvi", 2011, 2025), "ndvi_max": ("ndvi", 2011, 2025),
    "tmean_c": ("climate", 2011, 2025), "tmax_c": ("climate", 2011, 2025),
    "precip_mm": ("climate", 2011, 2025), "hot_days": ("climate", 2011, 2025),
    "jrc_monsoon_flood": ("jrc", 2011, 2022), "jrc_monsoon_obs": ("jrc", 2011, 2022),
    "jrc_flash_flood": ("jrc", 2011, 2021), "jrc_flash_obs": ("jrc", 2011, 2021),
    "s1_monsoon_flood": ("s1", 2016, 2025), "s1_flash_flood": ("s1", 2015, 2025),
}

# plausible ranges: a value outside stops the pipeline for inspection; nothing is altered automatically
RANGES = {
    "yield_t_ha": (1.0, 7.0), "area_ha": (0, 300000), "production_t": (0, 2_000_000),
    "ndvi_peak": (0.0, 1.0), "ndvi_max": (0.0, 1.0), "tmean_c": (10, 35), "tmax_c": (15, 40),
    "precip_mm": (0, 3000), "hot_days": (0, 92),
    "jrc_monsoon_flood": (0, 1), "jrc_flash_flood": (0, 1), "jrc_monsoon_obs": (0, 1),
    "jrc_flash_obs": (0, 1), "s1_monsoon_flood": (0, 1), "s1_flash_flood": (0, 1),
}

for name, e in EXTRACTS.items():
    print(f"{name:8s}", {w: window(2017, *spec) for w, spec in e["windows"].items()})

ndvi     {'season': ('2017-01-01', '2017-05-01')}
climate  {'season': ('2016-11-01', '2017-06-01'), 'hot': ('2017-03-01', '2017-06-01')}
jrc      {'monsoon': ('2016-06-01', '2016-11-01'), 'monsoon_dry': ('2016-01-01', '2016-04-01'), 'flash': ('2017-04-01', '2017-06-01'), 'flash_dry': ('2017-01-01', '2017-04-01')}
s1       {'monsoon': ('2016-06-01', '2016-11-01'), 'monsoon_dry': ('2016-01-01', '2016-04-01'), 'flash': ('2017-04-01', '2017-06-01'), 'flash_dry': ('2017-01-01', '2017-04-01')}


### 0e. District gazetteer

All sources are mapped to one list of the **64 present-day districts**. A source name is first compared with the
canonical names after removing case, spaces and punctuation; names that still do not match are looked up in an
explicit alias table of the variants that occur in the sources (older English spellings such as Jessore and Comilla,
and transliteration variants such as Sirajgonj). A name that matches neither stops the pipeline: it is never dropped
or matched by similarity.

In [5]:
CANONICAL = [
    "Bagerhat", "Bandarban", "Barguna", "Barishal", "Bhola", "Bogura",
    "Brahmanbaria", "Chandpur", "Chapai Nawabganj", "Chattogram", "Chuadanga",
    "Cox's Bazar", "Cumilla", "Dhaka", "Dinajpur", "Faridpur", "Feni",
    "Gaibandha", "Gazipur", "Gopalganj", "Habiganj", "Jamalpur", "Jashore",
    "Jhalokati", "Jhenaidah", "Joypurhat", "Khagrachhari", "Khulna",
    "Kishoreganj", "Kurigram", "Kushtia", "Lakshmipur", "Lalmonirhat",
    "Madaripur", "Magura", "Manikganj", "Meherpur", "Moulvibazar",
    "Munshiganj", "Mymensingh", "Naogaon", "Narail", "Narayanganj",
    "Narsingdi", "Natore", "Netrokona", "Nilphamari", "Noakhali", "Pabna",
    "Panchagarh", "Patuakhali", "Pirojpur", "Rajbari", "Rajshahi",
    "Rangamati", "Rangpur", "Satkhira", "Shariatpur", "Sherpur", "Sirajganj",
    "Sunamganj", "Sylhet", "Tangail", "Thakurgaon",
]

ALIASES = {
    "barisal": "Barishal", "bogra": "Bogura", "brahamanbaria": "Brahmanbaria",
    "chapainawabgonj": "Chapai Nawabganj", "chapinawabgonj": "Chapai Nawabganj",
    "nawabganj": "Chapai Nawabganj", "nawabgonj": "Chapai Nawabganj",
    "chattagram": "Chattogram", "chittagong": "Chattogram", "coxbazar": "Cox's Bazar",
    "comilla": "Cumilla", "gopalgonj": "Gopalganj", "habigonj": "Habiganj",
    "hobigonj": "Habiganj", "jessore": "Jashore", "jhalakathi": "Jhalokati",
    "jhallokati": "Jhalokati", "khagrachari": "Khagrachhari", "kishoregonj": "Kishoreganj",
    "kustia": "Kushtia", "laksmipur": "Lakshmipur", "manikgonj": "Manikganj",
    "maulavibazar": "Moulvibazar", "maulovibazar": "Moulvibazar", "maulvibazar": "Moulvibazar",
    "munsigonj": "Munshiganj", "mymenshing": "Mymensingh", "noagaon": "Naogaon",
    "narayangonj": "Narayanganj", "netrakona": "Netrokona", "panchagar": "Panchagarh",
    "perojpur": "Pirojpur", "sirajgonj": "Sirajganj", "sunamgonj": "Sunamganj",
}
_CANON_KEY = {re.sub(r"[^a-z]", "", c.lower()): c for c in CANONICAL}


def canon_district(name):
    # Canonical district name, or None if the name is not a district
    if name is None:
        return None
    key = re.sub(r"[^a-z]", "", str(name).lower())
    if not key:
        return None
    return _CANON_KEY.get(key) or ALIASES.get(key)


class DataContractError(Exception):
    # An input violates the data contract; the message starts with a check ID
    pass


print(len(CANONICAL), "districts,", len(ALIASES), "alias rules; e.g.", canon_district("Jessore"), canon_district("COMILLA"))

64 districts, 34 alias rules; e.g. Jashore Cumilla


The settings above are written out in full so that this notebook can be read on its own. The next cell confirms that
they are identical to the pipeline's single specification (`v2/src/schema.py` and `v2/src/common.py`), so the
notebook and the command-line pipeline cannot drift apart.

In [6]:
sys.path.insert(0, str(V2 / "src"))
import common as _pipeline_common
import schema as _pipeline_schema

assert EXTRACTS == _pipeline_schema.EXTRACTS, "extraction settings differ from schema.py"
assert COVERAGE == _pipeline_schema.COVERAGE and RANGES == _pipeline_schema.RANGES
assert CANONICAL == _pipeline_common.CANONICAL and ALIASES == _pipeline_common.ALIASES
print("Settings identical to the pipeline specification.")

Settings identical to the pipeline specification.


---
## 1. Bronze Layer — Raw Data Ingestion

The Bronze layer stores data **as received** from each source. Nothing is corrected by hand: rows that fail a check
are excluded and logged, and every check is recorded in a file that later steps verify.

### 1a. GADM v4.1 — District Boundaries

**Source:** Database of Global Administrative Areas (GADM), version 4.1, level 2 (districts).
**Role:** the spatial backbone: Earth Engine extracts are computed over these polygons, and the GADM identifier
`GID_2` is the stable district identifier (`district_id`) of the Gold tables.
**Licence:** free for academic use; redistribution is not permitted, so the shapefile is downloaded here.

In [7]:
GADM_URL = "https://geodata.ucdavis.edu/gadm/gadm4.1/shp/gadm41_BGD_shp.zip"
if not (SHP_DIR / "gadm41_BGD_2.shp").exists():
    SHP_DIR.mkdir(parents=True, exist_ok=True)
    zip_path = SHP_DIR / "gadm41_BGD_shp.zip"
    print("Downloading GADM v4.1 for Bangladesh ...")
    urllib.request.urlretrieve(GADM_URL, zip_path)
    zipfile.ZipFile(zip_path).extractall(SHP_DIR)

gadm = gpd.read_file(SHP_DIR / "gadm41_BGD_2.shp")
gadm["district"] = gadm.NAME_2.map(canon_district)
assert sorted(gadm.district) == sorted(CANONICAL), "GADM districts do not map one-to-one onto the gazetteer"
print(len(gadm), "district polygons, CRS", gadm.crs.to_string())
gadm[["GID_2", "NAME_1", "NAME_2", "district"]].head()

64 district polygons, CRS EPSG:4326


,GID_2,NAME_1,NAME_2,district
0,BGD.1.1_1,Barisal,Barguna,Barguna
1,BGD.1.2_1,Barisal,Barisal,Barishal
2,BGD.1.3_1,Barisal,Bhola,Bhola
3,BGD.1.4_1,Barisal,Jhalokati,Jhalokati
4,BGD.1.5_1,Barisal,Patuakhali,Patuakhali


### 1b. BBS Yearbooks — Table Extraction from the PDFs

**Source:** Bangladesh Bureau of Statistics, *Yearbook of Agricultural Statistics*, editions 2012–2025.
**Table:** "Estimates of Total Boro (Local + HYV + Hybrid)" by district. Its pages were located once in each edition
from the table titles and are recorded below.
**Method:** camelot in lattice mode reads the ruled table. Each printed line gives, for each of the two crop years in
the edition, five numbers: area in acres and hectares, yield in maunds per acre and tonnes per hectare, and
production in tonnes. A line is kept only if it has a name and 5 or 10 numbers (one or two crop years); leading
serial numbers are skipped. The crop years are read from the table header, and a line whose number of crop-year
blocks differs from the header stops the extraction: crop years are never assigned by position.

In [8]:
BBS_PAGES = {
    2012: "90,91", 2013: "92,93", 2014: "90,91", 2015: "94,95",
    2016: "96,97", 2017: "95,96", 2018: "96,97", 2019: "96,97",
    2020: "92,93", 2021: "98,99", 2022: "98,99", 2023: "99,100",
    2024: "96,97", 2025: "97,98",
}
assert BBS_PAGES == _pipeline_schema.BBS_PAGES
YEAR_RE = re.compile(r"(20\d\d)\s*-\s*(?:20)?(\d\d)(?!\d)")


def to_num(s):
    s = str(s).replace(",", "").replace("\n", "").strip()
    if re.fullmatch(r"-?\d+(\.\d+)?", s):
        return float(s)
    return None


def parse_table(df, book, page):
    # Rows of one camelot table: name, and five numbers per crop year
    header = " ".join(df.iloc[:3].astype(str).values.ravel())
    years = []
    for a, b in YEAR_RE.findall(header):
        cy = f"{a}-{b}"
        if cy not in years:
            years.append(cy)
    rows = []
    for _, r in df.iterrows():
        cells = [str(c).replace("\n", " ").strip() for c in r.tolist()]
        nums, name_parts = [], []
        for c in cells:
            v = to_num(c)
            if v is None:
                if c:
                    name_parts.append(c)
            elif name_parts or nums:            # skip leading serial numbers
                nums.append(v)
        name = re.sub(r"^\d+\s*", "", " ".join(name_parts)).strip()
        if not name or len(nums) not in (5, 10):
            continue
        blocks = [nums[i:i + 5] for i in range(0, len(nums), 5)]
        if len(blocks) != len(years):
            # Crop years are read from the table header and never assigned by position
            raise ValueError(f"{book} edition, page {page}: row '{name}' has {len(blocks)} crop-year block(s) "
                             f"but the table header gives {len(years)} crop year(s) {years}; check the table")
        for cy, (acre, ha, maund, tha, prod) in zip(years, blocks):
            rows.append(dict(book_year=book, page=page, raw_name=name, crop_year=cy,
                             area_acres=acre, area_ha=ha, yield_maund_acre=maund,
                             yield_t_ha=tha, production_t=prod))
    return rows


t0 = time.time()
parsed = []
for book, pages in BBS_PAGES.items():
    tables = camelot.read_pdf(str(PDF_DIR / f"bbs_agri_yearbook_{book}.pdf"), pages=pages, flavor="lattice")
    for t in tables:
        parsed += parse_table(t.df, book, t.page)
    print(f"{book} edition: pages {pages}, {len(tables)} tables")
d = pd.DataFrame(parsed)
print(f"\n{len(d):,} rows parsed in {time.time() - t0:.0f} s")

2012 edition: pages 90,91, 2 tables


2013 edition: pages 92,93, 2 tables


2014 edition: pages 90,91, 2 tables


2015 edition: pages 94,95, 2 tables


2016 edition: pages 96,97, 2 tables


...

2021 edition: pages 98,99, 2 tables


2022 edition: pages 98,99, 2 tables


2023 edition: pages 99,100, 2 tables


2024 edition: pages 96,97, 2 tables


2025 edition: pages 97,98, 2 tables

1,966 rows parsed in 60 s


### 1c. BBS — Row Classification and Unit Identities

**Row types.** Each parsed line is a district, a division or region, or the national total. The 2012 and 2013
editions group districts under former regions; five regions (Bandarban, Khagrachhari, Rangamati, Mymensingh and
Tangail) contain exactly one present-day district and print no district row beneath them, so the region row is that
district's printed estimate. These five rows are tagged and verified in 1d before they are used. A name that is not
recognised stops the pipeline.

**Unit identities.** With 1 acre = 0.404686 ha and 1 maund = 37.3242 kg, each row gives three relative discrepancies:

$$E_A = \frac{|0.404686\,A_{ac} - A_{ha}|}{A_{ha}}, \qquad
  E_Y = \frac{|0.0373242\,Y_{md}/0.404686 - Y_t|}{Y_t}, \qquad
  E_P = \frac{|A_{ha}\,Y_t - P|}{P}.$$

A row is accepted only if all three are at most 0.5 %: more than the error from rounding printed values, far less
than the discrepancy produced by a misread or shifted cell.

In [9]:
ACRE_HA, MAUND_T, TOL = 0.404686, 0.0373242, 0.005
SINGLE_TAG = " [single-district region]"
single = {"bandarbanregion": "Bandarban", "khagrachariregion": "Khagrachhari", "rangamatiregion": "Rangamati",
          "mymenshingregion": "Mymensingh", "tangailregion": "Tangail"}

key = d.raw_name.str.lower().str.replace(r"[^a-z]", "", regex=True)
is_single = d.book_year.isin([2012, 2013]) & key.isin(single)
d["district"] = d.raw_name.map(canon_district)
d.loc[is_single, "district"] = key[is_single].map(single)
d.loc[is_single, "raw_name"] = d.loc[is_single, "raw_name"] + SINGLE_TAG
name_l = d.raw_name.str.lower()
d["row_type"] = np.where(name_l.str.contains("bangladesh|national"), "national",
                 np.where(name_l.str.contains("division|divison|region") & ~is_single, "division",
                 np.where(d.district.notna(), "district", "unknown")))

d["qc_ha"] = (d.area_acres * ACRE_HA - d.area_ha).abs() / d.area_ha
d["qc_yield"] = (d.yield_maund_acre * MAUND_T / ACRE_HA - d.yield_t_ha).abs() / d.yield_t_ha
d["qc_prod"] = (d.area_ha * d.yield_t_ha - d.production_t).abs() / d.production_t
d["qc_pass"] = (d.qc_ha <= TOL) & (d.qc_yield <= TOL) & (d.qc_prod <= TOL)
d.to_csv(BBS_OUT / "bbs_boro_rows_all.csv", index=False)

unknown = d[d.row_type == "unknown"].raw_name.unique()
if len(unknown):
    raise ValueError(f"unrecognised row names in BBS tables: {list(unknown)}")
print("rows by type:", d.row_type.value_counts().to_dict())
print("district rows failing an identity:", int((~d[d.row_type == "district"].qc_pass).sum()))
d[d.district == "Rajshahi"].head(4)

rows by type: {'district': 1726, 'division': 218, 'national': 22}
district rows failing an identity: 4


,book_year,page,raw_name,crop_year,area_acres,area_ha,yield_maund_acre,yield_t_ha,production_t,district,row_type,qc_ha,qc_yield,qc_prod,qc_pass
142,2012,91,Rajshahi,2010-11,180307.0,72966.0,43.46,4.009,292510.0,Rajshahi,district,0.000024,0.000170,0.000037,True
143,2012,91,Rajshahi,2011-12,189045.0,76502.0,44.15,4.073,311571.0,Rajshahi,district,0.000024,0.000256,0.000069,True
306,2013,93,Rajshahi,2011-12,189045.0,76502.0,44.15,4.073,311571.0,Rajshahi,district,0.000024,0.000256,0.000069,True
307,2013,93,Rajshahi,2012-13,184627.0,74714.0,43.94,4.053,302834.0,Rajshahi,district,0.000026,0.000102,0.000060,True


### 1d. BBS — National Totals, Edition Comparison, Region Rows and the District-Year Table

1. **National totals.** For each edition and crop year, the sum of district production is compared with the printed
   national total (the comparison is meaningful only for complete tables of 64 districts).
2. **Edition comparison.** Each crop year from 2011-12 to 2023-24 is printed in two editions; the two rows of each
   district are compared on area and yield.
3. **Region rows (2012 and 2013 editions).** Each multi-district region row must equal the sum of the district rows
   listed under it, and each single-district region row must equal the district row printed for the same crop year
   (2012-13) in the district-structured 2014 edition. A sum of $k$ printed values may differ from a printed total by up
   to $0.5(k+1)$ because each value is rounded. The pipeline stops if any region row does not reconcile.
4. **District-year table.** One row per district and crop year, taken from the later edition among the rows that
   pass the identities, with the edition and page of every value.

A district printed twice in the same table, or a district row without a crop year, stops the run before these
checks: rows are never dropped or chosen silently.

In [10]:
nat = d[d.row_type == "national"][["book_year", "crop_year", "area_ha", "yield_t_ha", "production_t"]]
nat.to_csv(BBS_OUT / "bbs_boro_national.csv", index=False)
dist = d[d.row_type == "district"].copy()
# A district row without a crop year, or a district printed twice in one
# table, stops the run: rows are never dropped or chosen silently.
if dist.crop_year.isna().any():
    bad = dist[dist.crop_year.isna()]
    raise ValueError("district rows without a crop year: " + ", ".join(
        f"{r.district} ({int(r.book_year)} edition, page {int(r.page)})" for r in bad.head(5).itertuples()))
dup = dist[dist.duplicated(["book_year", "crop_year", "district"], keep=False)]
if len(dup):
    raise ValueError("district printed more than once in one table: " + ", ".join(
        f"{r.district} ({int(r.book_year)} edition, {r.crop_year}, page {int(r.page)})"
        for r in dup.head(6).itertuples()))

# 1. District sums against the printed national totals
s = dist.groupby(["book_year", "crop_year"]).agg(
    n_districts=("district", "nunique"), sum_prod=("production_t", "sum"),
    sum_area=("area_ha", "sum"), n_qc_fail=("qc_pass", lambda x: int((~x).sum())))
s = s.join(nat.set_index(["book_year", "crop_year"])[["production_t", "area_ha"]]
           .rename(columns={"production_t": "national_prod", "area_ha": "national_area"}))
s["prod_gap_pct"] = 100 * (s.sum_prod / s.national_prod - 1)
s.to_csv(BBS_OUT / "bbs_edition_checks.csv")

# 2. Years printed in two editions
keys = ["district", "crop_year"]
multi = dist[dist.duplicated(keys, keep=False)].sort_values(keys + ["book_year"])
diffs = []
for (dn, cy), g in multi.groupby(keys):
    a, b = g.iloc[0], g.iloc[-1]
    if abs(a.yield_t_ha - b.yield_t_ha) > 0.0005 or abs(a.area_ha - b.area_ha) > 0.5:
        diffs.append(dict(district=dn, crop_year=cy, early_book=a.book_year, late_book=b.book_year,
                          early_yield=a.yield_t_ha, late_yield=b.yield_t_ha,
                          early_area=a.area_ha, late_area=b.area_ha))
pd.DataFrame(diffs).to_csv(BBS_OUT / "bbs_edition_diffs.csv", index=False)

complete_tables = s[s.n_districts == 64]
print("complete tables:", len(complete_tables), "| with a printed national total:",
      int(complete_tables.national_prod.notna().sum()),
      "| largest gap: %.6f %%" % complete_tables.prod_gap_pct.abs().max())
print("district-years printed in two editions:", multi[keys].drop_duplicates().shape[0],
      "| differing:", len(diffs))
s.reset_index()[["book_year", "crop_year", "n_districts", "n_qc_fail", "sum_prod", "national_prod", "prod_gap_pct"]]

complete tables: 24 | with a printed national total: 18 | largest gap: 0.000021 %
district-years printed in two editions: 766 | differing: 6


,book_year,crop_year,n_districts,n_qc_fail,sum_prod,national_prod,prod_gap_pct
0,2012,2010-11,64,0,1.861678e+07,NaN,NaN
1,2012,2011-12,64,0,1.875921e+07,NaN,NaN
2,2013,2011-12,64,0,1.875921e+07,18759212.0,0.000005
3,2013,2012-13,64,4,1.877816e+07,18778154.0,0.000005
4,2014,2012-13,64,0,1.877816e+07,18778154.0,0.000005
5,2014,2013-14,64,0,1.900720e+07,19007206.0,-0.000016
6,2015,2013-14,63,0,1.832469e+07,19007206.0,-3.590817
7,2015,2014-15,63,0,1.846225e+07,19192164.0,-3.803167
8,2016,2014-15,64,0,1.919217e+07,19192164.0,0.000010
9,2016,2015-16,64,0,1.893758e+07,18937581.0,-0.000021


In [11]:
# 3. Region rows of the 2012 and 2013 editions
geo = []
for (b, cy), g in d[d.book_year.isin([2012, 2013])].groupby(["book_year", "crop_year"], sort=False):
    g = g.reset_index(drop=True)
    start = 0
    for i, r in g.iterrows():
        is_one = "single-district" in r.raw_name
        if (r.row_type == "division" and "region" in r.raw_name.lower()) or is_one:
            block = g.iloc[start:i]
            block = block[block.row_type == "district"]
            if not is_one and len(block):
                geo.append(dict(check="region = sum of listed districts", book_year=b, crop_year=cy,
                                unit=r.raw_name, districts=", ".join(block.district),
                                region_area_ha=r.area_ha, compare_area_ha=block.area_ha.sum(),
                                region_production_t=r.production_t,
                                compare_production_t=block.production_t.sum()))
            start = i + 1
later = d[(d.book_year == 2014) & (d.row_type == "district")].set_index(["district", "crop_year"])
for _, r in d[d.raw_name.str.contains("single-district")].iterrows():
    if (r.district, r.crop_year) in later.index:
        x = later.loc[(r.district, r.crop_year)]
        geo.append(dict(check="single-district region = 2014 district row", book_year=r.book_year,
                        crop_year=r.crop_year, unit=r.raw_name, districts=r.district,
                        region_area_ha=r.area_ha, compare_area_ha=x.area_ha,
                        region_production_t=r.production_t, compare_production_t=x.production_t))
geo = pd.DataFrame(geo)
tol = 0.5 * (geo.districts.str.count(",") + 2)           # 0.5 × (k + 1) for a sum of k rounded values
geo["match"] = ((geo.region_area_ha - geo.compare_area_ha).abs() <= tol) & \
               ((geo.region_production_t - geo.compare_production_t).abs() <= tol)
geo.to_csv(BBS_OUT / "bbs_region_checks.csv", index=False)
assert geo.match.all(), "region rows do not reconcile with district rows"
print(geo.groupby("check").match.agg(["size", "sum"]).to_string())
geo[geo.check.str.startswith("single")][["unit", "region_area_ha", "compare_area_ha", "region_production_t",
                                         "compare_production_t", "match"]]

                                            size  sum
check                                                
region = sum of listed districts              72   72
single-district region = 2014 district row     5    5


,unit,region_area_ha,compare_area_ha,region_production_t,compare_production_t,match
72,Bandarban Region [single-district region],3227.0,3227.0,9650.0,9650.0,True
73,Khagrachari Region [single-district region],10272.0,10272.0,33326.0,33326.0,True
74,Rangamati Region [single-district region],4788.0,4788.0,16623.0,16623.0,True
75,Mymenshing Region [single-district region],262040.0,262040.0,936015.0,936015.0,True
76,Tangail Region [single-district region],163636.0,163636.0,658469.0,658469.0,True


In [12]:
# 4. The district-year table: later edition among rows that pass the identities
fails = dist[~dist.qc_pass].copy()
fails["failed_checks"] = fails.apply(lambda r: ", ".join(
    n for n, c in [("area", r.qc_ha), ("yield", r.qc_yield), ("production", r.qc_prod)] if c > TOL), axis=1)
fails.to_csv(BBS_OUT / "bbs_qc_failures.csv", index=False)

good = dist[dist.qc_pass].sort_values("book_year")
bbs_final = good.drop_duplicates(keys, keep="last").copy()
bbs_final["harvest_year"] = bbs_final.crop_year.str[:4].astype(int) + 1
bbs_final = bbs_final.sort_values(["harvest_year", "district"])
bbs_final.to_csv(BBS_OUT / "bbs_boro_district_year.csv", index=False)

print("rows excluded by the identities:", len(fails), "| district-years:", len(bbs_final),
      "| districts:", bbs_final.district.nunique(), "| harvest years:", bbs_final.harvest_year.nunique())
display(fails[["book_year", "crop_year", "district", "yield_maund_acre", "yield_t_ha", "qc_yield", "failed_checks"]])
bbs_final[["district", "crop_year", "harvest_year", "area_ha", "yield_t_ha", "production_t", "book_year", "page"]].head()

rows excluded by the identities: 4 | district-years: 960 | districts: 64 | harvest years: 15


,book_year,crop_year,district,yield_maund_acre,yield_t_ha,qc_yield,failed_checks
225,2013,2012-13,Shariatpur,46.89,4.171,0.036841,yield
259,2013,2012-13,Magura,47.16,3.727,0.167043,yield
261,2013,2012-13,Narail,46.30,4.169,0.024286,yield
303,2013,2012-13,Sirajganj,46.48,4.157,0.031237,yield


,district,crop_year,harvest_year,area_ha,yield_t_ha,production_t,book_year,page
102,Bagerhat,2010-11,2011,45561.0,3.311,150872.0,2012,91
0,Bandarban,2010-11,2011,3295.0,2.871,9459.0,2012,90
118,Barguna,2010-11,2011,439.0,2.183,958.0,2012,91
80,Barishal,2010-11,2011,60520.0,3.765,227852.0,2012,91
86,Bhola,2010-11,2011,56213.0,3.484,195826.0,2012,91


### 1e. Google Earth Engine — Setup

The district polygons are uploaded to Earth Engine as a feature collection (simplified by 0.001° to keep requests
small). The image collections and masks used by the extraction functions are defined once:

* WorldCover v200 cropland (class 40) is averaged to the MODIS 250 m grid, and a cell counts as cropland if more than
  half of it is cropland;
* Sentinel-1 GRD scenes are restricted to IW mode with VV polarisation over Bangladesh;
* the SRTM slope mask removes slopes above 5° from radar water detection.

Earth Engine is initialised only when an extract has to be computed.

In [13]:
ee = None


def setup_ee():
    global ee, FC, BD, MODIS, CROPLAND, ERA, JRC, S1, SLOPE_OK
    if ee is not None:
        return
    import ee as _ee
    ee = _ee
    ee.Initialize(project=GEE_PROJECT)
    g = gadm[["district", "geometry"]].copy()
    g["geometry"] = g.geometry.simplify(0.001, preserve_topology=True)
    gj = json.loads(g.to_json())
    FC = ee.FeatureCollection([ee.Feature(ee.Geometry(f["geometry"]), {"district": f["properties"]["district"]})
                               for f in gj["features"]])
    BD = FC.geometry().bounds()
    MODIS = ee.ImageCollection(EXTRACTS["ndvi"]["collection"])
    CROPLAND = (ee.Image("ESA/WorldCover/v200/2021").eq(40).unmask(0)
                .reduceResolution(ee.Reducer.mean(), maxPixels=1024)
                .reproject(MODIS.first().projection()).gt(0.5))
    ERA = ee.ImageCollection(EXTRACTS["climate"]["collection"])
    JRC = ee.ImageCollection(EXTRACTS["jrc"]["collection"])
    S1 = (ee.ImageCollection(EXTRACTS["s1"]["collection"])
          .filter(ee.Filter.eq("instrumentMode", "IW"))
          .filter(ee.Filter.listContains("transmitterReceiverPolarisation", "VV"))
          .filterBounds(BD).select("VV"))
    SLOPE_OK = ee.Terrain.slope(ee.Image("USGS/SRTMGL1_003")).lt(EXTRACTS["s1"]["slope_max_deg"])
    print("Earth Engine initialised for project", GEE_PROJECT)


def reduce(img, scale, reducer=None, tile=8):
    # District means of every band of img over the GADM polygons
    reducer = reducer or ee.Reducer.mean()
    bands = img.bandNames().getInfo()
    if len(bands) == 1:
        # for a one-band image Earth Engine names the output after the reducer; name it after the band
        reducer = reducer.setOutputs(bands)
    res = img.reduceRegions(collection=FC, reducer=reducer, scale=scale, tileScale=tile)
    rows = res.select(["district"] + bands, retainGeometry=False).getInfo()
    return pd.DataFrame([f["properties"] for f in rows["features"]])

### 1f. Earth Engine Extraction Functions

Each function computes one product for one harvest year. Every variable is first computed for each pixel (a
seasonal mean, total, count or maximum, or for flood a classification as seasonally inundated or not) and then
averaged over each district polygon. NDVI and the climate variables are therefore district means of seasonal
aggregates, and the flood variables are the share of district area classified as seasonally inundated.

**Flood definition.** Water in any month of the flood window that was not water in the matching dry window
(January–March) is counted as seasonal inundation. The dry-season subtraction removes permanent rivers and water
bodies. JRC also records whether each pixel had at least one valid (cloud-free) observation.

In [14]:
def expected_columns(name, year):
    # Variables a product can provide for harvest year t, given its actual availability
    return [v for v, (src, y0, y1) in COVERAGE.items() if src == name and y0 <= year <= y1]


def ndvi(year):
    e = EXTRACTS["ndvi"]
    col = (MODIS.filterDate(*window(year, *e["windows"]["season"]))
           .map(lambda i: i.select("NDVI").multiply(0.0001).updateMask(i.select("SummaryQA").lte(1))))
    img = ee.Image.cat(col.mean().rename("ndvi_peak"), col.max().rename("ndvi_max"))
    return reduce(img.updateMask(CROPLAND), e["scale_m"])


def climate(year):
    e = EXTRACTS["climate"]
    season = ERA.filterDate(*window(year, *e["windows"]["season"]))
    tmean = season.select("temperature_2m").mean().subtract(273.15).rename("tmean_c")
    tmax = season.select("temperature_2m_max").mean().subtract(273.15).rename("tmax_c")
    prec = season.select("total_precipitation_sum").sum().multiply(1000).rename("precip_mm")
    hot = (ERA.filterDate(*window(year, *e["windows"]["hot"])).select("temperature_2m_max")
           .map(lambda i: i.subtract(273.15).gt(e["hot_threshold_c"])).sum().rename("hot_days"))
    return reduce(ee.Image.cat(tmean, tmax, prec, hot), e["scale_m"])


def _jrc_water(year, w):
    # Water in any month of window w, and whether each pixel was observed at least once
    offset, m0, m1 = w
    y = year + offset
    c = JRC.filter(ee.Filter.calendarRange(y, y, "year")).filter(ee.Filter.calendarRange(m0, m1, "month"))
    water = c.map(lambda i: i.eq(2)).max().unmask(0)
    seen = c.map(lambda i: i.gt(0)).max().unmask(0)
    return water, seen


def jrc_flood(year):
    e = EXTRACTS["jrc"]
    w = e["windows"]
    mon_w, mon_seen = _jrc_water(year, w["monsoon"])
    dry_prev, _ = _jrc_water(year, w["monsoon_dry"])
    bands = [mon_w.And(dry_prev.Not()).rename("jrc_monsoon_flood"), mon_seen.rename("jrc_monsoon_obs")]
    if "jrc_flash_flood" in expected_columns("jrc", year):      # JRC monthly ends in December 2021
        fl_w, fl_seen = _jrc_water(year, w["flash"])
        dry_cur, _ = _jrc_water(year, w["flash_dry"])
        bands += [fl_w.And(dry_cur.Not()).rename("jrc_flash_flood"), fl_seen.rename("jrc_flash_obs")]
    return reduce(ee.Image.cat(bands).clip(BD), e["scale_m"])


def _s1_water(start, end):
    e = EXTRACTS["s1"]
    vv = S1.filterDate(start, end).median().focal_median(e["focal_median_m"], "circle", "meters")
    return vv.lt(e["threshold_db"]).And(SLOPE_OK).unmask(0)


def _s1_monthly_any(year, w):
    offset, m0, m1 = w
    y = year + offset
    return ee.ImageCollection([_s1_water(*window(y, 0, m, m)) for m in range(m0, m1 + 1)]).max()


def s1_flood(year):
    e = EXTRACTS["s1"]
    w = e["windows"]
    bands = []
    if "s1_monsoon_flood" in expected_columns("s1", year):      # first complete Jun–Oct season is 2015
        mon = _s1_monthly_any(year, w["monsoon"])
        dry_prev = _s1_water(*window(year, *w["monsoon_dry"]))
        bands.append(mon.And(dry_prev.Not()).rename("s1_monsoon_flood"))
    fl = _s1_monthly_any(year, w["flash"])
    dry_cur = _s1_water(*window(year, *w["flash_dry"]))
    bands.append(fl.And(dry_cur.Not()).rename("s1_flash_flood"))
    return reduce(ee.Image.cat(bands).clip(BD), e["scale_m"])


JOBS = {"ndvi": (ndvi, HARVEST_YEARS), "climate": (climate, HARVEST_YEARS),
        "jrc": (jrc_flood, range(2011, 2023)), "s1": (s1_flood, range(2015, 2026))}
for name, (fn, years) in JOBS.items():
    print(f"{name:8s} harvest years {min(years)}–{max(years)}; 2015 variables: {expected_columns(name, 2015)}")

ndvi     harvest years 2011–2025; 2015 variables: ['ndvi_peak', 'ndvi_max']
climate  harvest years 2011–2025; 2015 variables: ['tmean_c', 'tmax_c', 'precip_mm', 'hot_days']
jrc      harvest years 2011–2022; 2015 variables: ['jrc_monsoon_flood', 'jrc_monsoon_obs', 'jrc_flash_flood', 'jrc_flash_obs']
s1       harvest years 2015–2025; 2015 variables: ['s1_flash_flood']


### 1g. Running the Extraction with a Cache and a Completeness Test

Each extract is saved as `<product>_<year>.csv`. An extract is used only if it is **complete**: it must contain all
64 districts and a non-missing value of every variable its product should provide for that harvest year. An
incomplete extract is recomputed, and an extract that is still incomplete is not saved. The date and duration of each
extraction are recorded.

With `REUSE_REPOSITORY_EXTRACTS = True` the extracts already in the repository are copied into the build cache first
(they pass the same completeness test). With `False`, every extract is computed from Earth Engine.

In [15]:
TIMES, DATES = GEE_OUT / "extraction_times.json", GEE_OUT / "extraction_dates.json"
if REUSE_REPOSITORY_EXTRACTS:
    repo_gee = V2 / "data" / "bronze" / "gee"
    for f in [*repo_gee.glob("*_20*.csv"), repo_gee / "extraction_times.json", repo_gee / "extraction_dates.json"]:
        shutil.copy2(f, GEE_OUT / f.name)
    print("copied", len(list(GEE_OUT.glob("*_20*.csv"))), "repository extracts into the build cache")


def complete(df, name, year):
    need = expected_columns(name, year)
    return len(df) == 64 and df.district.nunique() == 64 and all(c in df and df[c].notna().all() for c in need)


def cached(name, year, fn):
    f = GEE_OUT / f"{name}_{year}.csv"
    if f.exists():
        df = pd.read_csv(f)
        if complete(df, name, year):
            return df
        print(f"  {name} {year}: cached extract incomplete, re-extracting")
    setup_ee()
    for attempt in range(3):
        try:
            t0 = time.time()
            df = fn(year)
            df["harvest_year"] = year
            if not complete(df, name, year):
                raise ValueError(f"incomplete extract; expected {expected_columns(name, year)}, got {list(df.columns)}")
            df.to_csv(f, index=False)
            dt = time.time() - t0
            for path, value in [(TIMES, round(dt, 1)), (DATES, datetime.now(timezone.utc).strftime("%Y-%m-%d"))]:
                log = json.loads(path.read_text()) if path.exists() else {}
                log[f"{name}_{year}"] = value
                path.write_text(json.dumps(dict(sorted(log.items())) if path == DATES else log, indent=1))
            print(f"  {name} {year}: {len(df)} rows in {dt:.0f} s")
            return df
        except Exception as e:                      # transient Earth Engine errors
            print(f"  {name} {year}: attempt {attempt + 1} failed: {e}")
            time.sleep(10)
    raise RuntimeError(f"{name} {year} failed")


for name, (fn, years) in JOBS.items():
    frames = [cached(name, y, fn) for y in years]
    pd.concat(frames).to_csv(GEE_OUT / f"{name}_all.csv", index=False)
    print(f"{name:8s} {len(frames)} harvest years, {sum(len(x) for x in frames)} district-years")

copied 53 repository extracts into the build cache


ndvi     15 harvest years, 960 district-years


climate  15 harvest years, 960 district-years


jrc      12 harvest years, 768 district-years
s1       11 harvest years, 704 district-years


**Live check.** To show that the extraction functions reproduce the cached extracts, one extract (NDVI, harvest year
2016) is recomputed from Earth Engine now and compared with the cache. This needs an authenticated Earth Engine
account and takes about half a minute; it is skipped otherwise.

In [16]:
LIVE_DIFF = None
if CHECK_ONE_EXTRACT_LIVE:
    try:
        setup_ee()
        live = ndvi(2016).set_index("district").sort_index()
        cache = pd.read_csv(GEE_OUT / "ndvi_2016.csv").set_index("district").sort_index()
        diff = (live[["ndvi_peak", "ndvi_max"]] - cache[["ndvi_peak", "ndvi_max"]]).abs().max().max()
        LIVE_DIFF = dict(extract="ndvi_2016", districts=len(live), max_abs_diff=float(diff))
        print(f"NDVI 2016 recomputed for {len(live)} districts; largest difference from the cache: {diff:.2e}")
    except Exception as e:
        print("Live check skipped (Earth Engine not available):", str(e)[:150])

Earth Engine initialised for project bd-agri-climate


NDVI 2016 recomputed for 64 districts; largest difference from the cache: 3.33e-16


### 1h. Metadata Manifest

One row per source file and extract: the product and version, the exact date windows used for that harvest year,
the spatial scale, mask, method and parameters, the variables present, the number of rows, the extraction date, the
Earth Engine client version and the SHA-256 checksum. The validation report (Section 4) recomputes every checksum,
so a file that changes after the manifest is written is detected. Yearbook PDFs and GADM files are identified by
their checksums, which also lets a user confirm that their downloads are the files used.

In [17]:
WINDOW_VARS = {
    "ndvi": {"season": ["ndvi_peak", "ndvi_max"]},
    "climate": {"season": ["tmean_c", "tmax_c", "precip_mm"], "hot": ["hot_days"]},
    "jrc": {"monsoon": ["jrc_monsoon_flood"], "monsoon_dry": ["jrc_monsoon_flood"],
            "flash": ["jrc_flash_flood"], "flash_dry": ["jrc_flash_flood"]},
    "s1": {"monsoon": ["s1_monsoon_flood"], "monsoon_dry": ["s1_monsoon_flood"],
           "flash": ["s1_flash_flood"], "flash_dry": ["s1_flash_flood"]},
}
BBS_NAME = "BBS Yearbook of Agricultural Statistics, 2012–2025 editions"
GADM_NAME = "GADM v4.1, level 2"


def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


dates = json.loads(DATES.read_text()) if DATES.exists() else {}
manifest = []
for ed, pages in BBS_PAGES.items():
    f = PDF_DIR / f"bbs_agri_yearbook_{ed}.pdf"
    manifest.append(dict(kind="BBS yearbook PDF", file=f"data/bronze/bbs/pdfs/{f.name}", product=BBS_NAME,
                         version=f"{ed} edition", detail=f"Total Boro district table, PDF pages {pages}",
                         size_bytes=f.stat().st_size, sha256=sha256(f)))
for f in sorted(SHP_DIR.glob("gadm41_BGD_[02].*")):
    manifest.append(dict(kind="GADM boundaries", file=f"data/bronze/gadm_shapefiles/{f.name}", product=GADM_NAME,
                         version="4.1", detail="level 2 districts" if "_2." in f.name else "level 0 national outline",
                         size_bytes=f.stat().st_size, sha256=sha256(f)))
ee_version = version("earthengine-api")
for name, e in EXTRACTS.items():
    for f in sorted(GEE_OUT.glob(f"{name}_20*.csv")):
        year = int(f.stem.split("_")[1])
        df = pd.read_csv(f)
        present = [v for v, (src, _, _) in COVERAGE.items() if src == name and v in df.columns]
        wins = [f"{w} {window(year, *e['windows'][w])[0]} to {window(year, *e['windows'][w])[1]} (exclusive)"
                for w, vars_ in WINDOW_VARS[name].items() if any(v in present for v in vars_)]
        params = {k: v for k, v in e.items() if k.endswith(("_c", "_db", "_m", "_deg")) and k != "scale_m"}
        manifest.append(dict(kind="Earth Engine extract", file=f"v2/data/bronze/gee/{f.name}",
                             product=e["collection"], version=e["version"], harvest_year=year, bands=e["bands"],
                             windows="; ".join(wins), scale_m=e["scale_m"], mask=e["mask"], method=e["method"],
                             parameters="; ".join(f"{k}={v}" for k, v in params.items()),
                             variables=", ".join(present), rows=len(df), software=f"earthengine-api {ee_version}",
                             size_bytes=f.stat().st_size, sha256=sha256(f), extracted_utc=dates.get(f.stem, "")))
manifest = pd.DataFrame(manifest)
manifest.to_csv(BRONZE / "manifest.csv", index=False)
print(manifest.kind.value_counts().to_string())
manifest[manifest.kind == "Earth Engine extract"][["file", "windows", "parameters", "extracted_utc"]].tail(3)

kind
Earth Engine extract    53
BBS yearbook PDF        14
GADM boundaries         10


,file,windows,parameters,extracted_utc
74,v2/data/bronze/gee/s1_2023.csv,monsoon 2022-06-01 to 2022-11-01 (exclusive); ...,threshold_db=-18; focal_median_m=50; slope_max...,2026-10-25
75,v2/data/bronze/gee/s1_2024.csv,monsoon 2023-06-01 to 2023-11-01 (exclusive); ...,threshold_db=-18; focal_median_m=50; slope_max...,2026-10-25
76,v2/data/bronze/gee/s1_2025.csv,monsoon 2024-06-01 to 2024-11-01 (exclusive); ...,threshold_db=-18; focal_median_m=50; slope_max...,2026-10-25


---
## 2. Silver Layer — Input Contract, District Harmonisation and Validation

Before any table is built, every Bronze file is checked against a **data contract**. A violated rule raises an error
whose message starts with a check identifier and names the file, column and records involved; nothing is dropped or
corrected silently.

| Check ID | Rule |
|---|---|
| `[SCHEMA]` | every required column present; numeric columns contain only numbers (text such as "n/a" is an error, not a missing value) |
| `[MISSING]` | BBS area, yield and production present in every row; every district-year has a BBS value |
| `[DISTRICT]` | every district name maps to the gazetteer |
| `[KEY]` | one row per district and harvest year |
| `[RANGE]` | every value within its declared plausible range |

Each validated source is then written as a Silver Parquet table keyed on district and harvest year.

In [18]:
BRONZE_FILES = {
    "bbs/bbs_boro_rows_all.csv": ["book_year", "page", "raw_name", "crop_year", "area_acres", "area_ha",
                                  "yield_maund_acre", "yield_t_ha", "production_t", "district", "row_type",
                                  "qc_ha", "qc_yield", "qc_prod", "qc_pass"],
    "bbs/bbs_boro_district_year.csv": ["district", "crop_year", "harvest_year", "area_ha", "yield_t_ha",
                                       "production_t", "book_year", "page"],
    "bbs/bbs_boro_national.csv": ["book_year", "crop_year", "area_ha", "yield_t_ha", "production_t"],
    "gee/ndvi_all.csv": ["district", "harvest_year"], "gee/climate_all.csv": ["district", "harvest_year"],
    "gee/jrc_all.csv": ["district", "harvest_year"], "gee/s1_all.csv": ["district", "harvest_year"],
}
REQUIRED_NONNULL = {"bbs/bbs_boro_district_year.csv": ["district", "crop_year", "harvest_year", "area_ha",
                                                       "yield_t_ha", "production_t", "book_year", "page"]}
NUMERIC = ["harvest_year", "area_ha", "yield_t_ha", "production_t", "book_year", "page", *COVERAGE]
assert BRONZE_FILES == _pipeline_schema.BRONZE_FILES and REQUIRED_NONNULL == _pipeline_schema.REQUIRED_NONNULL


def records(df, mask, col=None, n=5):
    # Label up to n records for a diagnostic: district and year, and the offending value of col
    keys = [k for k in ("district", "harvest_year", "crop_year") if k in df.columns][:2]
    out = []
    for i, r in df[mask].head(n).iterrows():
        label = " ".join(str(r[k]) for k in keys) or f"row {i + 2}"
        out.append(f"{label} = {r[col]!r}" if col else label)
    return f"{int(mask.sum())} record(s): " + ", ".join(out)


def check_inputs():
    problems, missing_values = [], []
    for rel, cols in BRONZE_FILES.items():
        if rel == "bbs/bbs_boro_rows_all.csv":
            continue                                   # extraction log, not an input here
        f = BRONZE / rel
        if not f.exists():
            problems.append(f"{rel}: file missing")
            continue
        df = pd.read_csv(f, keep_default_na=False, na_values=[""])   # only an empty cell counts as missing
        for c in REQUIRED_NONNULL.get(rel, []):
            if c in df.columns and df[c].isna().any():
                missing_values.append(f"{rel}.{c} empty in {records(df, df[c].isna())}")
        need = list(cols)
        if rel.startswith("gee/"):
            need += [v for v, (src, _, _) in COVERAGE.items() if src == rel.split("/")[1].split("_")[0]]
        missing = [c for c in need if c not in df.columns]
        if missing:
            problems.append(f"{rel}: missing required columns {missing}")
        bad_type = [c for c in need if c in df.columns and c in NUMERIC and not pd.api.types.is_numeric_dtype(df[c])]
        if bad_type:
            where = "; ".join(f"{c} in {records(df, pd.to_numeric(df[c], errors='coerce').isna() & df[c].notna(), c)}"
                              for c in bad_type)
            problems.append(f"{rel}: non-numeric values in numeric columns {bad_type}: {where}")
    if problems:
        raise DataContractError("[SCHEMA] " + "; ".join(problems))
    if missing_values:
        raise DataContractError("[MISSING] " + "; ".join(missing_values))


def harmonise(df, name):
    mapped = df.district.map(canon_district)
    unknown = sorted(df.district[mapped.isna()].astype(str).unique())
    if unknown:
        raise DataContractError(f"[DISTRICT] {name}: unrecognised district names {unknown} in "
                                f"{records(df, mapped.isna())}")
    return df.assign(district=mapped)


def validate(df, name):
    for c, (lo, hi) in RANGES.items():
        if c in df:
            bad = df[df[c].notna() & ~df[c].between(lo, hi)]
            if len(bad):
                recs = ", ".join(f"{r.district} {r.harvest_year} = {getattr(r, c)}" for r in bad.head(5).itertuples())
                raise DataContractError(f"[RANGE] {name}.{c} outside [{lo}, {hi}] in {len(bad)} record(s): {recs}")
    dup = df[df.duplicated(["district", "harvest_year"], keep=False)]
    if len(dup):
        raise DataContractError(f"[KEY] {name}: duplicate (district, harvest_year) keys "
                                f"{sorted(set(zip(dup.district, dup.harvest_year)))}")


check_inputs()
bbs = harmonise(pd.read_csv(BBS_OUT / "bbs_boro_district_year.csv"), "bbs")
bbs = bbs[["district", "harvest_year", "crop_year", "area_ha", "yield_t_ha", "production_t", "book_year", "page"]] \
    .rename(columns={"book_year": "bbs_edition", "page": "bbs_page"})
nat_rows = pd.read_csv(BBS_OUT / "bbs_boro_national.csv")
nat_rows = (nat_rows.sort_values("book_year").drop_duplicates("crop_year", keep="last")
            .assign(harvest_year=lambda x: x.crop_year.str[:4].astype(int) + 1))

silver = {"bbs_boro": bbs}
for name in ["ndvi", "climate", "jrc", "s1"]:
    silver[name] = harmonise(pd.read_csv(GEE_OUT / f"{name}_all.csv"), name)
for name, df in silver.items():
    validate(df, name)
    df.to_parquet(SILVER / f"silver_{name}.parquet", index=False)
    print(f"silver_{name}.parquet: {len(df)} rows, {df.district.nunique()} districts, "
          f"harvest years {df.harvest_year.min()}–{df.harvest_year.max()}")

silver_bbs_boro.parquet: 960 rows, 64 districts, harvest years 2011–2025
silver_ndvi.parquet: 960 rows, 64 districts, harvest years 2011–2025
silver_climate.parquet: 960 rows, 64 districts, harvest years 2011–2025
silver_jrc.parquet: 768 rows, 64 districts, harvest years 2011–2022
silver_s1.parquet: 704 rows, 64 districts, harvest years 2015–2025


---
## 3. Gold Layer — DuckDB Analytical Tables

The Gold layer is a DuckDB database with eight tables:

| Table | Grain | Content |
|---|---|---|
| `dim_district` | district | GADM identifier (`district_id` = GID_2), name, division, area |
| `stg_bbs_boro`, `stg_ndvi`, `stg_climate`, `stg_jrc`, `stg_s1` | district × harvest year | the validated Silver tables |
| `fact_boro_national` | harvest year | national totals from the district sums, with the printed national row where one exists |
| `fact_boro_panel` | district × harvest year | the integrated panel: BBS area, yield and production with edition and page, and every covariate for its source period |

District areas are computed in UTM zone 46N and rounded to 0.001 km², so they do not depend on the last digit of the
installed geometry libraries. The panel is built from the BBS table and left-joined to the covariates, so a missing
covariate stays missing; a district-year without a BBS value stops the build (`[MISSING]`).

In [19]:
g = gadm.copy()
g["division"] = g.NAME_1.replace({"Barisal": "Barishal", "Chittagong": "Chattogram"})
g["area_km2"] = (g.to_crs(32646).area / 1e6).round(3)
dim = pd.DataFrame(g[["GID_2", "district", "division", "area_km2"]]).rename(columns={"GID_2": "district_id"})

if GOLD_DB.exists():
    GOLD_DB.unlink()
con = duckdb.connect(str(GOLD_DB))
for name, df in silver.items():
    con.register("tmp", df)
    con.execute(f"CREATE OR REPLACE TABLE stg_{name} AS SELECT * FROM tmp")
    con.unregister("tmp")
con.register("tmp", dim)
con.execute("CREATE OR REPLACE TABLE dim_district AS SELECT * FROM tmp")
con.unregister("tmp")

con.register("tmp", nat_rows[["harvest_year", "area_ha", "yield_t_ha", "production_t"]])
con.execute('''
    CREATE OR REPLACE TABLE fact_boro_national AS
    SELECT b.harvest_year, any_value(b.crop_year) AS crop_year,
           sum(b.area_ha) AS area_ha, sum(b.production_t) AS production_t,
           sum(b.production_t) / sum(b.area_ha) AS yield_t_ha,
           any_value(t.production_t) AS printed_national_production_t,
           any_value(t.yield_t_ha) AS printed_national_yield_t_ha
    FROM stg_bbs_boro b LEFT JOIN tmp t USING (harvest_year)
    GROUP BY b.harvest_year ORDER BY b.harvest_year
''')
con.unregister("tmp")

con.execute('''
    CREATE OR REPLACE TABLE fact_boro_panel AS
    SELECT d.district_id, b.district, d.division, d.area_km2, b.harvest_year, b.crop_year,
           b.yield_t_ha, b.area_ha, b.production_t, b.bbs_edition, b.bbs_page,
           n.ndvi_peak, n.ndvi_max,
           c.tmean_c, c.tmax_c, c.precip_mm, c.hot_days,
           j.jrc_monsoon_flood, j.jrc_flash_flood, j.jrc_monsoon_obs, j.jrc_flash_obs,
           s.s1_monsoon_flood, s.s1_flash_flood
    FROM stg_bbs_boro b
    JOIN dim_district d USING (district)
    LEFT JOIN stg_ndvi n USING (district, harvest_year)
    LEFT JOIN stg_climate c USING (district, harvest_year)
    LEFT JOIN stg_jrc j USING (district, harvest_year)
    LEFT JOIN stg_s1 s USING (district, harvest_year)
    ORDER BY b.district, b.harvest_year
''')
panel = con.execute("SELECT * FROM fact_boro_panel").df()

full = pd.MultiIndex.from_product([sorted(CANONICAL), HARVEST_YEARS], names=["district", "harvest_year"])
absent = full.difference(pd.MultiIndex.from_frame(panel[["district", "harvest_year"]]))
if len(absent):
    raise DataContractError(f"[MISSING] fact_boro_panel: no BBS value for {len(absent)} district-year(s): "
                            + ", ".join(f"{d_} {y}" for d_, y in absent[:5]))
assert len(panel) == 64 * 15 and not panel.duplicated(["district", "harvest_year"]).any()

display(pd.DataFrame([(t, con.execute(f"SELECT count(*) FROM {t}").fetchone()[0])
                      for t in con.execute("SHOW TABLES").df().name], columns=["table", "rows"]))
con.close()

,table,rows
0,dim_district,64
1,fact_boro_national,15
2,fact_boro_panel,960
3,stg_bbs_boro,960
4,stg_climate,960
5,stg_jrc,768
6,stg_ndvi,960
7,stg_s1,704


**Coverage of the panel.** Each covariate has values for exactly the harvest years its product covers (Section 0d).

In [20]:
cov = pd.DataFrame([{"variable": v, "product": src, "harvest years": f"{a}–{b}",
                     "district-years with values": int(panel[v].notna().sum()), "expected": 64 * (b - a + 1)}
                    for v, (src, a, b) in COVERAGE.items()])
display(cov)
panel[panel.district == "Sunamganj"][["district_id", "district", "harvest_year", "yield_t_ha", "area_ha", "production_t",
                                      "bbs_edition", "bbs_page", "ndvi_peak", "tmean_c", "jrc_flash_flood",
                                      "s1_flash_flood"]].head(8)

,variable,product,harvest years,district-years with values,expected
0,ndvi_peak,ndvi,2011–2025,960,960
1,ndvi_max,ndvi,2011–2025,960,960
2,tmean_c,climate,2011–2025,960,960
3,tmax_c,climate,2011–2025,960,960
4,precip_mm,climate,2011–2025,960,960
5,hot_days,climate,2011–2025,960,960
6,jrc_monsoon_flood,jrc,2011–2022,768,768
7,jrc_monsoon_obs,jrc,2011–2022,768,768
8,jrc_flash_flood,jrc,2011–2021,704,704
9,jrc_flash_obs,jrc,2011–2021,704,704


,district_id,district,harvest_year,yield_t_ha,area_ha,production_t,bbs_edition,bbs_page,ndvi_peak,tmean_c,jrc_flash_flood,s1_flash_flood
900,BGD.7.3_1,Sunamganj,2011,3.448,190662.0,657326.0,2012,90,0.526193,21.895020,0.001990,NaN
901,BGD.7.3_1,Sunamganj,2012,3.407,188786.0,643271.0,2013,92,0.499222,21.681482,0.105823,NaN
902,BGD.7.3_1,Sunamganj,2013,3.367,190525.0,641455.0,2014,90,0.508718,21.529829,0.213900,NaN
903,BGD.7.3_1,Sunamganj,2014,3.488,186282.0,649668.0,2015,94,0.532060,21.805404,0.059945,NaN
904,BGD.7.3_1,Sunamganj,2015,3.611,192704.0,695856.0,2016,96,0.538783,21.935441,0.303766,0.078941
905,BGD.7.3_1,Sunamganj,2016,3.619,189367.0,685226.0,2017,95,0.541776,21.890446,0.438974,0.344019
906,BGD.7.3_1,Sunamganj,2017,3.613,54388.0,196500.0,2018,96,0.489219,21.906284,0.372031,0.447092
907,BGD.7.3_1,Sunamganj,2018,3.671,201037.0,738001.0,2019,97,0.544489,21.832146,0.197041,0.061779


In [21]:
with duckdb.connect(str(GOLD_DB), read_only=True) as con:
    display(con.execute("SELECT harvest_year, crop_year, round(area_ha) AS area_ha, round(production_t) AS production_t, "
                        "round(yield_t_ha, 3) AS yield_t_ha, printed_national_production_t, "
                        "printed_national_yield_t_ha FROM fact_boro_national").df())

,harvest_year,crop_year,area_ha,production_t,yield_t_ha,printed_national_production_t,printed_national_yield_t_ha
0,2011,2010-11,4770337.0,18616780.0,3.903,NaN,NaN
1,2012,2011-12,4810025.0,18759213.0,3.900,18759212.0,3.900
2,2013,2012-13,4760055.0,18778155.0,3.945,18778154.0,3.945
3,2014,2013-14,4790309.0,19007198.0,3.968,19007206.0,3.968
4,2015,2014-15,4840224.0,19192166.0,3.965,19192164.0,3.965
5,2016,2015-16,4772576.0,18937577.0,3.968,18937581.0,3.968
6,2017,2016-17,4475879.0,18013748.0,4.025,18013749.0,4.025
7,2018,2017-18,4859367.0,19575815.0,4.028,19575819.0,4.028
8,2019,2018-19,4788276.0,19560548.0,4.085,19560546.0,4.085
9,2020,2019-20,4762131.0,19645331.0,4.125,19645334.0,4.125


---
## 4. Validation Report

The full validation report of the paper (`v2/src/s03b_validate.py`) is now run **on the data built in this
notebook**: the environment variable `BD_AGRI_V2_DIR` points it to `build/` instead of the pipeline's own folder. It
checks seven categories (structural, relational, temporal, value, geographic, cross-source and lineage), writes
`build/outputs/validation_report.csv`, the district crosswalk and the data dictionary, and stops on any FAIL.

In [22]:
env = dict(os.environ, BD_AGRI_V2_DIR=str(BUILD), PYTHONIOENCODING="utf-8")
p = subprocess.run([sys.executable, "-W", "ignore", "s03b_validate.py"], cwd=V2 / "src", env=env,
                   capture_output=True, text=True, encoding="utf-8", errors="replace")
print("validation step exit code:", p.returncode)
report = pd.read_csv(BUILD / "outputs" / "validation_report.csv")
print(len(report), "checks:", report.status.value_counts().to_dict())
display(report.groupby(["category", "status"]).size().unstack(fill_value=0))
report[report.status != "PASS"][["category", "check", "observed", "status"]]

validation step exit code: 0
108 checks: {'PASS': 107, 'INFO': 1}


status,INFO,PASS
category,,
cross-source,1,7
geographic,0,13
lineage,0,3
relational,0,39
structural,0,15
temporal,0,14
value,0,16


,category,check,observed,status
100,cross-source,"unit identities (area, yield, production) with...","4 of 1726 rows failed, excluded",INFO


In [23]:
display(pd.read_csv(BUILD / "outputs" / "tables" / "table_district_matching.csv").set_index("Source").T)
xw = pd.read_csv(BUILD / "mappings" / "district_crosswalk.csv")
xw[xw.rule != "exact"].head(10)

Source,BBS yearbooks,Earth Engine extract,GADM v4.1
Records,1726,3392,64
Distinct spellings,99,65,64
Exact,60,64,54
Normalised,4,0,1
Alias rule,30,1,9
Region row,5,0,0
Unmatched,0,0,0
Districts reached,64,64,64


,source,source_name,canonical_district,n_records,editions,rule,district_id
2,BBS yearbooks,Bandarban Region,Bandarban,4,2012–2013,single-district region row (verified against 2...,BGD.2.1_1
4,BBS yearbooks,Barisal,Barishal,10,2012–2016,alias rule,BGD.1.2_1
7,BBS yearbooks,Bogra,Bogura,10,2012–2016,alias rule,BGD.5.1_1
9,BBS yearbooks,Brahmanbari a,Brahmanbaria,2,2015–2015,"normalised (case, spacing, punctuation)",BGD.2.2_1
12,BBS yearbooks,Chapai Nawabganj,Chapai Nawabganj,12,2020–2025,"normalised (case, spacing, punctuation)",BGD.5.5_1
13,BBS yearbooks,Chapai Nawabgonj,Chapai Nawabganj,4,2018–2019,alias rule,BGD.5.5_1
14,BBS yearbooks,Chapinawabgonj,Chapai Nawabganj,2,2014–2014,alias rule,BGD.5.5_1
15,BBS yearbooks,Nawabgonj,Chapai Nawabganj,10,2012–2017,alias rule,BGD.5.5_1
16,BBS yearbooks,Chattagram,Chattogram,2,2017–2017,alias rule,BGD.2.4_1
18,BBS yearbooks,Chittagong,Chattogram,10,2012–2016,alias rule,BGD.2.4_1


---
## 5. Comparison with the Command-Line Pipeline

The command-line pipeline (`python v2/run_all.py`) produces the same layers in `v2/data/`. The cell below compares
every Bronze and Silver file and every Gold table built in this notebook with the pipeline's: files byte for byte,
DuckDB tables value by value, and the Gold panel by its SHA-256 checksum.

In [24]:
def file_sha(p):
    return hashlib.sha256(Path(p).read_bytes()).hexdigest()


rows = []
for rel in ["bbs/bbs_boro_rows_all.csv", "bbs/bbs_boro_district_year.csv", "bbs/bbs_boro_national.csv",
            "bbs/bbs_edition_checks.csv", "bbs/bbs_edition_diffs.csv", "bbs/bbs_region_checks.csv",
            "bbs/bbs_qc_failures.csv", "gee/ndvi_all.csv", "gee/climate_all.csv", "gee/jrc_all.csv",
            "gee/s1_all.csv", "manifest.csv"]:
    rows.append(("Bronze", rel, file_sha(BRONZE / rel) == file_sha(V2 / "data" / "bronze" / rel)))
for f in sorted(SILVER.glob("*.parquet")):
    rows.append(("Silver", f.name, pd.read_parquet(f).equals(pd.read_parquet(V2 / "data" / "silver" / f.name))))
with duckdb.connect(str(GOLD_DB), read_only=True) as a, \
        duckdb.connect(str(V2 / "data" / "gold" / "bangladesh_agri_v2.duckdb"), read_only=True) as b:
    for t in a.execute("SHOW TABLES").df().name:
        rows.append(("Gold", t, a.execute(f"SELECT * FROM {t}").df().equals(b.execute(f"SELECT * FROM {t}").df())))
cmp = pd.DataFrame(rows, columns=["layer", "file or table", "identical to the pipeline"])
panel_same = _pipeline_common.panel_sha256(GOLD_DB) == _pipeline_common.panel_sha256(_pipeline_common.V2_GOLD)
print(f"{int(cmp.iloc[:, 2].sum())} of {len(cmp)} files and tables identical")
print("Gold panel SHA-256 identical:", panel_same)

# Saved for the manuscript, which quotes these results (tied to the panel they were produced from)
summary = dict(date_utc=datetime.now(timezone.utc).strftime("%Y-%m-%d"),
               compared=len(cmp), identical=int(cmp.iloc[:, 2].sum()),
               by_layer=cmp.groupby("layer").size().to_dict(), panel_identical=bool(panel_same),
               panel_sha256=_pipeline_common.panel_sha256(GOLD_DB), validation=report.status.value_counts().to_dict(),
               live_check=LIVE_DIFF)
(V2 / "docs" / "notebook_rebuild_summary.json").write_text(json.dumps(summary, indent=1), encoding="utf-8")
cmp

25 of 25 files and tables identical
Gold panel SHA-256 identical: True


,layer,file or table,identical to the pipeline
0,Bronze,bbs/bbs_boro_rows_all.csv,True
1,Bronze,bbs/bbs_boro_district_year.csv,True
2,Bronze,bbs/bbs_boro_national.csv,True
3,Bronze,bbs/bbs_edition_checks.csv,True
4,Bronze,bbs/bbs_edition_diffs.csv,True
5,Bronze,bbs/bbs_region_checks.csv,True
6,Bronze,bbs/bbs_qc_failures.csv,True
7,Bronze,gee/ndvi_all.csv,True
8,Bronze,gee/climate_all.csv,True
9,Bronze,gee/jrc_all.csv,True


---
## Summary

This notebook built the complete dataset of the paper from the original sources:

* **Bronze:** 1,726 district rows parsed from the 14 BBS yearbook editions and checked against unit identities,
  printed national totals, a second edition and the region rows; 53 Earth Engine district extracts (NDVI, ERA5-Land,
  JRC and Sentinel-1) with a completeness test; a manifest with the SHA-256 of every file;
* **Silver:** one validated table per source after the input contract, district harmonisation and range and key
  checks;
* **Gold:** a DuckDB database with the district-year panel of 960 BBS-observed values (64 districts × harvest years
  2011–2025) and every covariate for its source period.

The validation report raised no failures, and every file and table is identical to the output of the command-line
pipeline. The analyses, figures, failure tests and reproducibility tests of the paper are run from these data by
`v2/run_all.py` and the scripts in `v2/tests/`.